Script: this piece of code loads and processes precipitation files for multiple inputs for use later in plotting scripts

Notes: needs to be run individually for each model - check the boxes that have a *change me* tag on the top. 

For each model the tas and pr fields with the ensemble mean removed. 
DJFerem

In [1]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 

In [2]:
#set up the data directory and load in lon + lat + time dimensions

outputdirb='/glade/campaign/cgd/cas/nmaher/canesm5_lens/Amon/pr/' 
outputdir2='/glade/work/nmaher/SEOF_output/'
model='CanESM5-SSP585'

#option for merging files
ds_fx1 = xr.open_dataset(outputdirb+'pr_mon_CanESM5_historical_r10i1p2f1_g025.nc')
ds_fx2 = xr.open_dataset(outputdirb+'pr_mon_CanESM5_ssp585_r10i1p2f1_g025.nc')


ds_fx = xr.merge([ds_fx1, ds_fx2])

lon = ds_fx.lon
lat = ds_fx.lat
time = ds_fx.time

In [3]:
#*change me*
#set up list of files for each member
filesb = natsorted(os.listdir(outputdirb))
filesb = [s for s in filesb if "historical" in s and "i1p2f1" in s and "_g025.nc" in s]

files2b = natsorted(os.listdir(outputdirb))
files2b = [s for s in files2b if "ssp585" in s and "i1p2f1" in s and "_g025.nc" in s]

n = 25#len(filesb) #25 for canesm5 #10 IPSL
ne = np.empty(n)
for ii in range(n):
        ne[ii] = ii
print(n)              

25


In [4]:
#loop through ensemble members and load data
pr_all = np.empty((n,len(time),len(lat),len(lon)))
pr_all = xr.DataArray(pr_all, coords=[ne, time, lat, lon], dims=["member", "time", "lat", "lon"])

for ii in range(n):
        filenameH = outputdirb+filesb[ii]
        filenameS = outputdirb+files2b[ii]
        ds_memberH = xr.open_dataset(filenameH)
        ds_memberS = xr.open_dataset(filenameS)
        ds_member=xr.merge([ds_memberH, ds_memberS])
        pr = ds_member.pr
        pr_all[ii,:,:,:] = np.squeeze(pr.values)
        
        
        #loop through ensemble members and load data
zg_all = np.empty((n,len(time),len(lat),len(lon)))
zg_all = xr.DataArray(zg_all, coords=[ne, time, lat, lon], dims=["member", "time", "lat", "lon"])


In [5]:
#select season
pr_DJF_full = pr_all.where(pr_all['time.season'] == 'DJF')


del pr
import gc
collected = gc.collect()
 

In [6]:
#take seasonal mean for masked and full regions
pr_DJF_full = pr_DJF_full.rolling(min_periods=3, center=True, time=3).mean()

# make annual mean


In [7]:
pr_DJF_full = pr_DJF_full.groupby('time.year').mean('time')


In [8]:
#set up the input and the full
pr_DJF_2_full=pr_DJF_full[:,1:,:,:]
pr_DJF_2_full=pr_DJF_2_full.values


In [10]:
#save DJF emeanremoved zg

pr_DJF_2e_full = pr_DJF_2_full - np.ma.average(pr_DJF_2_full,axis=0)[np.newaxis,:,:,:]
eof_T='_prDJFerem'
np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=pr_DJF_2e_full.data, mask=pr_DJF_2e_full.mask)
